# 04 - Baseline B3: árbol y bosque aleatorio

La búsqueda y evaluación de finalistas ya están cerradas. La primera celda prepara este runtime de forma independiente y este cuaderno presenta los CSV versionados y la configuración común seleccionada; no ejecuta Optuna ni reentrena modelos. Los artefactos `.joblib` no están en `results/baseline3/` y deben buscarse en Drive antes de proponer un ajuste.

In [ ]:
# Cada cuaderno prepara su propio runtime de Colab; no depende de 00_arranque ni de otra sesión.
from pathlib import Path
import sys, json, time, subprocess
from datetime import datetime, timezone
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo fetch origin 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo checkout 04MainAIModelTrainingAndDevelopment
    !git -C /content/repo pull --ff-only origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / 'src') not in sys.path: sys.path.insert(0, str(REPO_DIR / 'src'))
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp.ingest import load_days
from ptcg_wp.features import feature_columns
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b3 import fit_b3_candidates_val
DATA_DIR = Path('/content/datos')
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(p.exists() for p in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(p.exists() for p in esperados), 'Faltan tablas procesadas; revisa Kaggle.'
schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62
SEED = 2026
SCENARIOS = ('A', 'B', 'C')
DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
RUN_INFO = {'commit': COMMIT, 'working_tree_dirty': DIRTY, 'seed': SEED, 'run_id_utc': RUN_ID}
RESULTS_DIR = REPO_DIR / 'results' / 'baseline3'
print('Tablas listas:', len(list(DATA_DIR.glob('estados_*.parquet'))), 'días | commit:', COMMIT,
      '| cambios locales:', DIRTY, '| corrida UTC:', RUN_ID)


In [ ]:
for escenario in ('A', 'B', 'C'):
    ruta = RESULTS_DIR / f'b3_val_{escenario}.csv'
    print(f'Escenario {escenario}')
    if ruta.exists():
        datos = pd.read_csv(ruta)
        display(datos.round(4))
    else:
        print('No está disponible:', ruta)


## Artefactos y límites

Los CSV de validación documentan los reajustes con todo `train`, pero esta carpeta no contiene los modelos serializados. No se repite la búsqueda y no se usa `test`. La selección se basó en valores observados, sin bootstrap emparejado ni afirmación de significancia estadística.

## Recuperar y verificar modelos B3

Tras montar Drive, busca solo el artefacto de la configuración finalista seleccionada para cada escenario, verifica hiperparámetros y lo copia a `export_repo/baseline3`. No ajusta modelos. El campo de escenario y la versión se muestran para revisar su procedencia.

In [ ]:
from pathlib import Path
import sys, shutil, hashlib, joblib
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
BUSCAR_MODELOS_B3 = False
if BUSCAR_MODELOS_B3:
    dest = root / 'export_repo' / 'baseline3'; dest.mkdir(parents=True, exist_ok=True)
    esperado = {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 20,
                'n_estimators': 25, 'max_features': 0.5}
    for escenario in ('A', 'B', 'C'):
        nombre = f'b3_{escenario}_B3_bosque_log_loss.joblib'
        encontrados = list((root / 'b3').rglob(nombre))
        if len(encontrados) != 1:
            print(nombre, ': se esperaba 1 archivo; encontrados:', encontrados); continue
        src = encontrados[0]; item = joblib.load(src)
        assert item.get('escenario') == escenario and len(item['features']) == 62
        parametros = item['model'].get_params()
        assert all(parametros[k] == v for k, v in esperado.items()), parametros
        assert item.get('version_sklearn') == '1.7.2'
        target = dest / nombre
        if target.exists():
            print('Ya existe; no se sobrescribe:', target); continue
        shutil.copy2(src, target)
        print('Copiado:', target, '| SHA256:', hashlib.file_digest(target.open('rb'), 'sha256').hexdigest(),
              '| sklearn:', item.get('version_sklearn'))


## Generar solo el modelo B3 finalista si falta en Drive

Ejecuta primero la celda de recuperación. Si no encuentra el `.joblib` seleccionado, reajusta únicamente el bosque finalista con los hiperparámetros acordados y el `train` propio del escenario; luego evalúa `val` y guarda el artefacto. No corre la búsqueda ni ajusta árboles u otras candidatas.

In [ ]:
import time, json
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b3 import fit_b3_candidates_val
GENERAR_B3_FINALISTA_SI_FALTA = False
if GENERAR_B3_FINALISTA_SI_FALTA:
    drive_root = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
    parametros = {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 20,
                  'n_estimators': 25, 'max_features': 0.5}
    for escenario in ('A', 'B', 'C'):
        nombre = f'b3_{escenario}_B3_bosque_log_loss.joblib'
        existentes = list((drive_root / 'b3').rglob(nombre))
        if existentes:
            print(f'{escenario}: ya existe el artefacto; no se entrena:', existentes)
            continue
        salida = drive_root / 'b3' / 'finalista_seleccionada_refit' / RUN_ID / f'escenario_{escenario}'
        salida.mkdir(parents=True, exist_ok=False)
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES))
        candidata = pd.DataFrame([{'escenario': escenario, 'modelo': 'B3_bosque',
            'seleccionada_por': 'log_loss', 'trial': 1, **parametros}])
        inicio = time.perf_counter()
        metricas = fit_b3_candidates_val(stream, FEATURES, candidata,
            max_train_games=None, seed=2026, n_jobs=2, max_ram_gib=10.0,
            cache_dir='/content', output_dir=salida, verbose=True)
        registro = {**RUN_INFO, 'escenario': escenario, 'baseline': 'B3',
                    'configuracion': parametros, 'segundos': time.perf_counter()-inicio}
        (salida / 'registro.json').write_text(json.dumps(registro, indent=2))
        display(metricas)
        print('Modelo guardado:', salida / nombre)


## Descargar resultados B3

Empaqueta los modelos finales exportados y, si se generaron en esta corrida, los manifiestos y métricas del reajuste finalista.
Ejecuta esta celda después de terminar las secciones que quieras conservar.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import shutil

def agregar_a_zip(archivo_zip, ruta, raiz_drive, carpeta_zip):
    ruta = Path(ruta)
    if not ruta.exists():
        print('No existe, se omite:', ruta)
        return 0
    n = 0
    for item in ruta.rglob('*'):
        if item.is_file():
            destino = Path(carpeta_zip) / item.relative_to(raiz_drive)
            archivo_zip.write(item, destino.as_posix(), compress_type=ZIP_DEFLATED, compresslevel=1)
            n += 1
    return n
raiz = DRIVE_ROOT.parent
zip_path = Path('/content') / f'resultados_B3_{RUN_ID}.zip'
with ZipFile(zip_path, 'w') as zf:
    cantidad = 0
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'export_repo' / 'baseline3', raiz, 'Applied-AI-Agent-Project')
    cantidad += agregar_a_zip(zf, DRIVE_ROOT / 'b3' / 'finalista_seleccionada_refit' / RUN_ID, raiz, 'Applied-AI-Agent-Project')
assert cantidad > 0, 'No hay modelos B3 exportados; ejecuta la recuperación o el reajuste de los finalistas.'
print('Archivos:', cantidad, '| ZIP:', zip_path, '| tamaño MiB:', round(zip_path.stat().st_size / 2**20, 1))
files.download(str(zip_path))
